LAB 4 (Week 6?)

In [21]:
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import StandardScaler
import numpy as np


In [22]:
# load data

df = pd.read_csv('data/pima-indians-diabetes.data', skiprows=2, header=None)

X = df.iloc[:, :-1] # everything except last column
y = df.iloc[:, -1] # last col
y = np.where(y == 0, -1, 1) # tanh uses -1/+1

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42
)

# scale the values
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# verify shape
print(X_train.shape)
print(X_test.shape)

(460, 8)
(308, 8)


In [ ]:
def tanh(x):
    x = np.clip(x, -500, 500)
    return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

def tanh_derivative(tanh_output):
    return 1 - tanh_output**2

LR = 0.1
hidden_size = 5
output_size = 1

target_success = 0.85
max_epochs = 5000
patience = 100
min_improvement = 0.0   # use 0.001 if you want tiny improvements to count

np.random.seed(42)
input_size = X_train.shape[1]

W1 = np.random.randn(input_size, hidden_size) * 0.01
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * 0.01
b2 = np.zeros((1, output_size))

allLoss = []
allAccuracy = []

best_success = 0
epochs_without_improvement = 0

for epoch in range(max_epochs):
    epoch_loss = 0
    correct = 0

    for i in range(len(X_train)):
        x_i = X_train[i].reshape(1, -1)
        y_i = np.array([[y_train[i]]])

        # forward pass
        hidden_linear = x_i @ W1 + b1
        hidden_output = tanh(hidden_linear)

        output_linear = hidden_output @ W2 + b2
        predicted_output = tanh(output_linear)

        # loss
        sample_loss = np.mean((predicted_output - y_i) ** 2)
        epoch_loss += sample_loss

        # classification
        predicted_label = 1 if predicted_output.item() >= 0 else -1
        if predicted_label == y_i.item():
            correct += 1

        # backward pass
        delta_output = (predicted_output - y_i) * tanh_derivative(predicted_output)
        delta_hidden = (delta_output @ W2.T) * tanh_derivative(hidden_output)

        # update
        W2 = W2 - LR * (hidden_output.T @ delta_output)
        b2 = b2 - LR * delta_output

        W1 = W1 - LR * (x_i.T @ delta_hidden)
        b1 = b1 - LR * delta_hidden

    current_loss = epoch_loss / len(X_train)
    success_rate = correct / len(X_train)

    allLoss.append(current_loss)
    allAccuracy.append(success_rate)

    print(f"Epoch {epoch+1}: Loss = {current_loss:.6f}, Success Rate = {success_rate:.4f}")

    # check improvement
    if success_rate > best_success + min_improvement:
        best_success = success_rate
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    # stop only if:
    # - success rate has reached target
    # - it has plateaued
    if best_success >= target_success and epochs_without_improvement >= patience:
        print(f"Stopped: success rate plateaued at {best_success:.4f} for {patience} epochs")
        break
else:
    print("Stopped: reached max epochs")

Epoch 1: Loss = 0.773931, Success Rate = 0.7000
Epoch 2: Loss = 0.715640, Success Rate = 0.7435
Epoch 3: Loss = 0.692945, Success Rate = 0.7435
Epoch 4: Loss = 0.678297, Success Rate = 0.7478
Epoch 5: Loss = 0.671849, Success Rate = 0.7587
Epoch 6: Loss = 0.666453, Success Rate = 0.7696
Epoch 7: Loss = 0.664609, Success Rate = 0.7674
Epoch 8: Loss = 0.663457, Success Rate = 0.7739
Epoch 9: Loss = 0.662451, Success Rate = 0.7761
Epoch 10: Loss = 0.661546, Success Rate = 0.7783
Epoch 11: Loss = 0.660985, Success Rate = 0.7739
Epoch 12: Loss = 0.661293, Success Rate = 0.7783
Epoch 13: Loss = 0.662514, Success Rate = 0.7783
Epoch 14: Loss = 0.663925, Success Rate = 0.7717
Epoch 15: Loss = 0.665366, Success Rate = 0.7717
Epoch 16: Loss = 0.666657, Success Rate = 0.7739
Epoch 17: Loss = 0.667603, Success Rate = 0.7739
Epoch 18: Loss = 0.668157, Success Rate = 0.7717
Epoch 19: Loss = 0.668216, Success Rate = 0.7696
Epoch 20: Loss = 0.667481, Success Rate = 0.7717
Epoch 21: Loss = 0.665635, Su

In [25]:
correct = 0

for i in range(len(X_test)):
    x_i = X_test[i].reshape(1, -1)
    y_i = y_test[i]

    hidden_linear = x_i @ W1 + b1
    hidden_output = my_tanh(hidden_linear)

    output_linear = hidden_output @ W2 + b2
    predicted_output = my_tanh(output_linear)

    predicted_label = 1 if predicted_output.item() >= 0 else -1

    if predicted_label == y_i:
        correct += 1

test_accuracy = correct / len(X_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Accuracy: 0.7695
